https://github.com/bmoscon/cryptofeed

In [23]:
from market_data import MarketDataCollector

In [24]:
collector = MarketDataCollector(
    exchange="BINANCE",
    symbol="BTC-USDT",
)

In [25]:
await collector.collect(30)

Started BINANCE BTC-USDT
Stopping feed...
Flushing storage...


ArrowKeyError: No type extension with name arrow.py_extension_type found

In [2]:
from cryptofeed import FeedHandler
from cryptofeed.exchanges import Coinbase

print("Cryptofeed works")

Cryptofeed works


In [3]:
from collections import deque

import pandas as pd

from cryptofeed import FeedHandler
from cryptofeed.defines import TRADES, L2_BOOK
from cryptofeed.exchanges import Coinbase, Kraken

In [5]:
async def trade_callback(trade, receipt_timestamp):

    trade_data.append({
        "exchange": trade.exchange,
        "symbol": trade.symbol,
        "side": trade.side,
        "price": float(trade.price),
        "amount": float(trade.amount),

        # Timestamp supplied by exchange
        "exchange_timestamp": trade.timestamp,

        # Timestamp when cryptofeed received it
        "receipt_timestamp": receipt_timestamp
    })

In [6]:
async def book_callback(book, receipt_timestamp):

    # Ignore incomplete books
    if not len(book.book.bids) or not len(book.book.asks):
        return

    # index(0) gives best price level
    bid_price, bid_size = book.book.bids.index(0)
    ask_price, ask_size = book.book.asks.index(0)

    bid_price = float(bid_price)
    ask_price = float(ask_price)
    bid_size = float(bid_size)
    ask_size = float(ask_size)

    mid = (bid_price + ask_price) / 2

    spread = ask_price - bid_price

    spread_bps = (
        spread / mid
    ) * 10_000

    book_data.append({
        "exchange": book.exchange,
        "symbol": book.symbol,

        "bid": bid_price,
        "bid_size": bid_size,

        "ask": ask_price,
        "ask_size": ask_size,

        "mid": mid,
        "spread": spread,
        "spread_bps": spread_bps,

        "exchange_timestamp": book.timestamp,
        "receipt_timestamp": receipt_timestamp
    })

In [7]:
fh = FeedHandler()

fh.add_feed(
    Coinbase(
        symbols=["BTC-USD"],
        channels=[TRADES, L2_BOOK],
        callbacks={
            TRADES: trade_callback,
            L2_BOOK: book_callback
        }
    )
)

fh.add_feed(
    Kraken(
        symbols=["BTC-USD"],
        channels=[TRADES, L2_BOOK],
        callbacks={
            TRADES: trade_callback,
            L2_BOOK: book_callback
        }
    )
)

In [10]:
import asyncio

async def collect(seconds=30):
    print(f"Collecting data for {seconds} seconds...")

    # Start Cryptofeed in Jupyter's existing event loop
    task = asyncio.create_task(
        fh.run_async(
            install_signal_handlers=False
        )
    )

    await asyncio.sleep(seconds)

    print("Stopping feeds...")

    # Ask Cryptofeed to stop gracefully
    fh.request_stop()

    # Wait for shutdown to complete
    await task

    print("Done.")

In [11]:
await collect(30)

Stopping feeds...
Done.


In [12]:
# Cell 2
trade_data = deque(maxlen=250_000)
book_data = deque(maxlen=250_000)

In [14]:
# Cell 3
async def trade_callback(trade, receipt_timestamp):
    trade_data.append({
        "exchange": trade.exchange,
        "symbol": trade.symbol,
        "side": trade.side,
        "price": float(trade.price),
        "amount": float(trade.amount),
        "exchange_timestamp": trade.timestamp,
        "receipt_timestamp": receipt_timestamp
    })


async def book_callback(book, receipt_timestamp):

    if not len(book.book.bids) or not len(book.book.asks):
        return

    bid, bid_size = book.book.bids.index(0)
    ask, ask_size = book.book.asks.index(0)

    bid = float(bid)
    ask = float(ask)
    bid_size = float(bid_size)
    ask_size = float(ask_size)

    mid = (bid + ask) / 2
    spread = ask - bid

    book_data.append({
        "exchange": book.exchange,
        "symbol": book.symbol,

        "bid": bid,
        "bid_size": bid_size,

        "ask": ask,
        "ask_size": ask_size,

        "mid": mid,
        "spread": spread,

        "spread_bps": (
            spread / mid
        ) * 10_000,

        "exchange_timestamp": book.timestamp,
        "receipt_timestamp": receipt_timestamp
    })

In [15]:
# Cell 4
fh = FeedHandler()

fh.add_feed(
    Coinbase(
        symbols=["BTC-USD"],
        channels=[TRADES, L2_BOOK],
        callbacks={
            TRADES: trade_callback,
            L2_BOOK: book_callback
        }
    )
)

fh.add_feed(
    Kraken(
        symbols=["BTC-USD"],
        channels=[TRADES, L2_BOOK],
        callbacks={
            TRADES: trade_callback,
            L2_BOOK: book_callback
        }
    )
)

In [16]:
# Cell 5
import asyncio

async def collect(seconds=30):

    print(f"Collecting for {seconds} seconds...")

    task = asyncio.create_task(
        fh.run_async(
            install_signal_handlers=False
        )
    )

    await asyncio.sleep(seconds)

    fh.request_stop()

    await task

    print("Done.")

In [17]:
# Cell 6
await collect(30)

Done.


In [18]:
print("Trades:", len(trade_data))
print("Book updates:", len(book_data))

Trades: 208
Book updates: 2498


In [19]:
trades = pd.DataFrame(trade_data)
books = pd.DataFrame(book_data)

display(trades.head())
display(books.head())

,exchange,symbol,side,price,amount,exchange_timestamp,receipt_timestamp
0,KRAKEN,BTC-USD,buy,84444.50,5.640600e-04,1.790483e+09,1.790483e+09
1,COINBASE,BTC-USD,buy,84438.43,2.600000e-07,1.790483e+09,1.790483e+09
2,COINBASE,BTC-USD,buy,84438.43,2.000000e-08,1.790483e+09,1.790483e+09
3,COINBASE,BTC-USD,buy,84438.43,1.100000e-07,1.790483e+09,1.790483e+09
4,COINBASE,BTC-USD,buy,84438.43,7.000000e-08,1.790483e+09,1.790483e+09


,exchange,symbol,bid,bid_size,ask,ask_size,mid,spread,spread_bps,exchange_timestamp,receipt_timestamp
0,KRAKEN,BTC-USD,84444.4,0.001652,84444.5,1.830661,84444.45,0.1,0.011842,1.790483e+09,1.790483e+09
1,KRAKEN,BTC-USD,84444.4,0.001652,84444.5,1.830661,84444.45,0.1,0.011842,1.790483e+09,1.790483e+09
2,KRAKEN,BTC-USD,84444.4,0.001652,84444.5,1.830661,84444.45,0.1,0.011842,1.790483e+09,1.790483e+09
3,KRAKEN,BTC-USD,84444.4,0.001652,84444.5,1.830661,84444.45,0.1,0.011842,1.790483e+09,1.790483e+09
4,KRAKEN,BTC-USD,84444.4,0.001652,84444.5,1.830661,84444.45,0.1,0.011842,1.790483e+09,1.790483e+09


In [22]:
trades[trades['exchange'] == 'KRAKEN']

,exchange,symbol,side,price,amount,exchange_timestamp,receipt_timestamp
0,KRAKEN,BTC-USD,buy,84444.5,0.000564,1.790483e+09,1.790483e+09
115,KRAKEN,BTC-USD,buy,84444.5,0.000197,1.790483e+09,1.790483e+09
132,KRAKEN,BTC-USD,sell,84444.4,0.000444,1.790483e+09,1.790483e+09
133,KRAKEN,BTC-USD,sell,84444.4,0.000100,1.790483e+09,1.790483e+09
134,KRAKEN,BTC-USD,sell,84444.0,0.009115,1.790483e+09,1.790483e+09
135,KRAKEN,BTC-USD,sell,84443.9,0.000200,1.790483e+09,1.790483e+09
136,KRAKEN,BTC-USD,sell,84443.9,0.000200,1.790483e+09,1.790483e+09
137,KRAKEN,BTC-USD,sell,84440.5,0.000137,1.790483e+09,1.790483e+09
138,KRAKEN,BTC-USD,sell,84440.5,0.001249,1.790483e+09,1.790483e+09
139,KRAKEN,BTC-USD,sell,84440.5,0.000444,1.790483e+09,1.790483e+09
